# Phase 3 — Planning & Decomposition

**Goal**: Some questions are too big for one tool call. *"Compare the top 3 countries in 2010 versus 2011"* is really **two** lookups plus a comparison. In this phase the orchestrator first sends the question to a **PlannerAgent**, which breaks it into a list of simple sub-tasks. The orchestrator then runs each sub-task with the DataAgent and stitches the results together.

In Phase 2 the pipeline was *fixed* (DataAgent -> Critic, always). Here the steps are **decided at runtime** by the planner — the system adapts to the question.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Decomposition** | Breaking one big question into several small, independently-answerable ones. |
| **PlannerAgent** | A worker whose only job is to *plan* — it returns a list of sub-tasks, it does not answer anything itself. |
| **Plan** | The list of sub-tasks. Here it is a JSON list of short questions. |
| **Execution loop** | The orchestrator runs the DataAgent once per sub-task, then combines the answers. |
| **Dynamic control flow** | The number of steps is not hard-coded — it depends on what the planner returns. |

## Acceptance criteria
1. The PlannerAgent produces a plan with **>= 2** sub-tasks
2. The orchestrator runs the DataAgent **once per sub-task** — `trace.n_delegations >= 3` (1 planner + 2+ data)
3. The final combined answer matches the pandas ground truth
4. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [36]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

[OK] Working dir: /Users/ilynn/Desktop/Personal/2026/AI_Project/Multi_Agents Project/orchestrator
[OK] Dev model: claude-haiku-4-5-20251001


In [37]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
# Phase 3 adds parse_plan to the machinery you already know:
from orchestrator.agents import AgentRun, run_agent, parse_plan
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

Loaded 1,067,371 rows
Dev model: claude-haiku-4-5-20251001


## 2. Ground truth — a multi-step question

This phase's question can't be answered in one lookup: **"Compare the top 3 countries by revenue in 2010 versus 2011."** That's two separate top-3 lists. We compute both with pandas so we know the right answer.

In [38]:
def top_3_countries(year):
    """Top 3 countries by revenue for one year — plain pandas."""
    yearly = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == year)]
    grouped = yearly.groupby("Country")["revenue"].sum()
    top = grouped.sort_values(ascending=False).head(3)
    return top.reset_index()

gt_2010 = top_3_countries(2010)

gt_2011 = top_3_countries(2011)

print("Top 3 countries by revenue, 2010:")
print(gt_2010)
print()
print("Top 3 countries by revenue, 2011:")
print(gt_2011)

# Every country name that should appear in the final combined answer
expected_countries = sorted(set(gt_2010["Country"]) | set(gt_2011["Country"]))
print()
print("Expected countries in the final answer:", expected_countries)

Top 3 countries by revenue, 2010:
          Country      revenue
0  United Kingdom  8707529.003
1            EIRE   370911.340
2     Netherlands   262365.750

Top 3 countries by revenue, 2011:
          Country      revenue
0  United Kingdom  8254828.984
1     Netherlands   276661.860
2            EIRE   273420.700

Expected countries in the final answer: ['EIRE', 'Netherlands', 'United Kingdom']


## 3. Rebuild the retail tool + MCP server (recap)

Same `query_retail` tool as Phases 1-2 — the DataAgent needs it. Just run it.

In [39]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Use this for any 'top N' question about products, "
        "countries, or customers. Arguments: year (optional), country (optional "
        "- OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2010"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])
print("[OK] Retail tool + MCP server ready")

[OK] Retail tool + MCP server ready


## 4. The DataAgent (worker — recap from Phase 2)

The DataAgent is unchanged: a worker that answers ONE data question with the `query_retail` tool. In this phase it gets called once per sub-task. Just run the cell.

In [40]:
data_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer the question you are given "
        "using the query_retail tool. Call the tool with the correct filters, "
        "then present the result as a short, clear markdown table. "
        "Only filter by country if the question explicitly names a country. "
        "Never invent numbers - report exactly what the tool returns."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] DataAgent configured")

[OK] DataAgent configured


## 5. The PlannerAgent

The **PlannerAgent** is the new idea in Phase 3. It does NOT answer the question. It reads a big question and breaks it into a **list of small sub-questions** — each one simple enough for the DataAgent to answer in a single tool call.

For the orchestrator to *use* the plan, the planner's reply must be **machine-readable**. The `parse_plan()` helper (in `agents.py`) looks for a **JSON list** inside the reply — e.g.

```json
["What were the top 3 countries by revenue in 2010?", "What were the top 3 countries by revenue in 2011?"]
```

So your prompt must make the planner output exactly that: a JSON list of strings.

**TODO 1**: write the PlannerAgent's system prompt.

In [41]:
planner_system_prompt = (
    # ----- TODO 1: write the PlannerAgent's system prompt ----------------
    # The planner BREAKS DOWN a question. It does NOT answer it and does
    # NOT use any tools.
    #
    # Your prompt MUST tell the planner to:
    #   - split the user's question into the smallest set of independent
    #     sub-questions needed to answer it
    #   - make each sub-question self-contained (the DataAgent will see it
    #     ALONE, with no other context)
    #   - reply with ONLY a JSON list of strings, nothing else
    #     (e.g. ["sub-question 1", "sub-question 2"])
    #
    # Write 3-5 sentences. Imagine briefing a project manager who splits
    # work into tickets.
   """You are the planning agent for a retail analytics system. Every question you receive is about a retail transactions dataset (sales by country, product, customer, and year).

Your job is to take the incoming question and break it into small, independent sub-questions for the next agent to handle. Make each sub-question self-contained — the next agent sees it alone, with no other context. Never ask for clarification — always produce a plan.

For example, "Compare the top 3 countries by revenue in 2010 versus 2011" breaks down into:
  1. What are the top 3 countries by revenue in 2010?
  2. What are the top 3 countries by revenue in 2011?

Reply with ONLY a JSON list of strings, nothing else — e.g. ["sub-question 1", "sub-question 2"]."""

    
    # ---------------------------------------------------------------------
)

planner_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=planner_system_prompt,
    # The planner has NO tools - it only thinks about how to split the work.
    max_turns=2,
)
print("[OK] PlannerAgent configured")

[OK] PlannerAgent configured


## 6. Watch the planner break a question down

Before wiring it into the orchestrator, see the planner work. We give it the complex question and print the plan `parse_plan()` extracts.

A good plan here has **2 sub-tasks** — one for 2010, one for 2011.

In [42]:
demo_question = "Compare the top 3 countries by revenue in 2010 versus 2011."

planner_run = await run_agent("PlannerAgent", demo_question, planner_options)
print("----- Planner raw reply -----")
print(planner_run.answer)

plan = parse_plan(planner_run.answer)
print()
print(f"----- Parsed plan ({len(plan)} sub-tasks) -----")
for i, step in enumerate(plan, start=1):
    print(f"  {i}. {step}")

assert len(plan) >= 2, "Planner did not return >=2 sub-tasks - strengthen your TODO 1 prompt."
print()
print("[OK] Planner produced a usable multi-step plan")

----- Planner raw reply -----
```json
[
  "What are the top 3 countries by revenue in 2010?",
  "What are the top 3 countries by revenue in 2011?"
]
```

----- Parsed plan (2 sub-tasks) -----
  1. What are the top 3 countries by revenue in 2010?
  2. What are the top 3 countries by revenue in 2011?

[OK] Planner produced a usable multi-step plan


## 7. The Orchestrator — plan, then execute each step


The orchestrator now works in three stages:

1. **Plan** — send the question to the PlannerAgent, get a list of sub-tasks
2. **Execute** — run the DataAgent once for *each* sub-task
3. **Combine** — stitch the sub-answers into one final answer

Stage 2 is a **loop whose length depends on the plan** — that's the dynamic control flow this phase is about.

**TODO 2**: fill in the execution loop — run the DataAgent on each sub-task.

In [ ]:
async def orchestrate_with_plan(question, planner_options, data_options):
    """Plan-and-execute orchestrator: plan the question, run each sub-task, combine."""
    runs = []
    n_delegations = 0

    # --- Stage 1: PlannerAgent breaks the question into sub-tasks ---
    planner_run = await run_agent("PlannerAgent", question, planner_options)
    runs.append(planner_run)
    n_delegations += 1
    plan = parse_plan(planner_run.answer)
    print(f"[plan] {len(plan)} sub-tasks")

    # --- Stage 2: run the DataAgent once per sub-task ---
    step_answers = []
    # ----- TODO 2: loop over `plan` and delegate each step to the DataAgent ---
    for step in plan:
        data_run = await run_agent("DataAgent", step, data_options)
        runs.append(data_run)
        n_delegations += 1
        step_answers.append(data_run.answer)
    # --------------------------------------------------------------------------

    # --- Stage 3: combine the sub-answers into one final answer ---
    final_parts = []
    for i in range(len(step_answers)):
        step = plan[i] if i < len(plan) else f"step {i + 1}"
        answer = step_answers[i]
        final_parts.append(f"### Sub-task {i + 1}: {step}\n{answer}")
    final_answer = "\n\n".join(final_parts)

    return {"answer": final_answer, "plan": plan,
            "runs": runs, "n_delegations": n_delegations}

print("[OK] Orchestrator defined")

## 8. Run the orchestrator

**TODO 3**: write the multi-step business question. It should genuinely need decomposition — ask the planner to **compare the top 3 countries by revenue in 2010 versus 2011**.

In [44]:
# ----- TODO 3: write your multi-step business question -----
QUESTION = "ask to compare the top 3 countries by revenue in 2010 versus 2011"

# -----------------------------------------------------------

trace = Trace(model=DEV_MODEL, question=QUESTION)

with Timer() as t:
    result = await orchestrate_with_plan(QUESTION, planner_options, data_options)

# Roll every worker's usage into the single Trace for this run
for r in result["runs"]:
    trace.input_tokens        += r.input_tokens
    trace.output_tokens       += r.output_tokens
    trace.cached_input_tokens += r.cached_input_tokens
    trace.n_tool_calls        += r.n_tool_calls
trace.n_delegations = result["n_delegations"]
trace.latency_ms    = t.elapsed_ms
trace.answer        = result["answer"]
trace.compute_cost()

print("\n----- Plan -----")
for i, step in enumerate(result["plan"], start=1):
    print(f"  {i}. {step}")
print("\n----- Final combined answer -----")
print(result["answer"])
print(f"\nDelegations: {trace.n_delegations} | Tool calls: {trace.n_tool_calls}")

[plan] 2 sub-tasks

----- Plan -----
  1. What are the top 3 countries by revenue in 2010?
  2. What are the top 3 countries by revenue in 2011?

----- Final combined answer -----
### Sub-task 1: What are the top 3 countries by revenue in 2010?
Here's a comparison of the top 3 countries by revenue in 2010 versus 2011:

| Rank | 2010 Country | 2010 Revenue | 2011 Country | 2011 Revenue |
|------|---|---|---|---|
| 1 | United Kingdom | $8,707,529.00 | United Kingdom | $8,254,828.98 |
| 2 | EIRE | $370,911.34 | Netherlands | $276,661.86 |
| 3 | Netherlands | $262,365.75 | EIRE | $273,420.70 |

**Key Insights:**
- **United Kingdom** maintained the top position both years but experienced a decline in revenue (down ~$452,700)
- **Netherlands and EIRE swapped positions**, with EIRE dropping from #2 to #3 despite only a slight revenue decrease (~$97,491)
- **Netherlands** grew slightly (up ~$14,296) and moved up to #2 position in 2011

### Sub-task 2: What are the top 3 countries by revenue in

In [45]:
result['answer']

"### Sub-task 1: What are the top 3 countries by revenue in 2010?\nHere's a comparison of the top 3 countries by revenue in 2010 versus 2011:\n\n| Rank | 2010 Country | 2010 Revenue | 2011 Country | 2011 Revenue |\n|------|---|---|---|---|\n| 1 | United Kingdom | $8,707,529.00 | United Kingdom | $8,254,828.98 |\n| 2 | EIRE | $370,911.34 | Netherlands | $276,661.86 |\n| 3 | Netherlands | $262,365.75 | EIRE | $273,420.70 |\n\n**Key Insights:**\n- **United Kingdom** maintained the top position both years but experienced a decline in revenue (down ~$452,700)\n- **Netherlands and EIRE swapped positions**, with EIRE dropping from #2 to #3 despite only a slight revenue decrease (~$97,491)\n- **Netherlands** grew slightly (up ~$14,296) and moved up to #2 position in 2011\n\n### Sub-task 2: What are the top 3 countries by revenue in 2011?\nHere's a comparison of the top 3 countries by revenue:\n\n| Rank | 2010 Country | 2010 Revenue | 2011 Country | 2011 Revenue |\n|------|---|---|---|---|\n| 1

## 9. Verify (acceptance criteria)

In [46]:
expected_countries

['EIRE', 'Netherlands', 'United Kingdom']

In [47]:
# ----- TODO 4: write the assertion -----
# Build `missing` - the list of countries from `expected_countries` (computed
# in the ground-truth cell) whose name does NOT appear in result["answer"].
# Same pattern as Phase 1 and 2: a for-loop that collects what's missing.


missing =[]
for country in expected_countries:
    if country not in result['answer']:
        missing.append(country)
# ---------------------------------------

trace.passed = (missing == [])
print(f"Missing countries: {missing}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Final answer missed: {missing}"

# Phase 3 acceptance criteria
assert len(result["plan"]) >= 2, \
    f"Expected a plan with >=2 sub-tasks, got {len(result['plan'])}"
assert trace.n_delegations >= 3, \
    f"Expected >=3 delegations (1 planner + 2+ data), got {trace.n_delegations}"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - plan made, steps executed, trace saved")

Missing countries: []
Passed: True

[OK] Acceptance criteria met - plan made, steps executed, trace saved


## 10. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer in your own words.

1. **Why decompose**: the DataAgent has a `query_retail` tool that can answer one lookup at a time. Why can't it answer "compare 2010 vs 2011" in a single call? What does the planner give you that a single agent does not?

   `query_retail` is one filtered groupby — it can produce a 2010 top-list *or* a 2011 top-list, never both side-by-side. A single agent without a planner could still loop and call the tool twice, but it has to *invent* that strategy on the fly. The planner externalizes that strategy into a structured artifact (the JSON list) — now the orchestrator can count steps, run them in parallel, log them, retry one without redoing the others. You've turned a hidden internal monologue into something you can engineer around.

2. **Planner vs DataAgent**: the planner never touches the data and never calls a tool. So what *is* its job? Why is it useful to have an agent that only plans?

   Its job is **task decomposition** — translate fuzzy English ("compare X vs Y") into a list of crisp, self-contained sub-questions. Useful as a standalone agent because (a) it can use a smarter (or differently-prompted) model than the cheap executor — pay for thinking once, then run many cheap executions, and (b) separation of concerns: the planner can be unit-tested on its plans without ever spinning up the data layer.

3. **Dynamic control flow**: in Phase 2 the number of steps was fixed. In Phase 3 it depends on the plan. If the planner returns 4 sub-tasks instead of 2, how many times does the DataAgent run? Where in the code is that decided?

   **4 times.** Decided by the `for step in plan:` loop inside `orchestrate_with_plan` — the length of `plan` is whatever the planner returned, and the loop body runs once per element. The orchestrator code is the same for 2 steps or 20; the *behavior* changes because the data (the plan) changes.

4. **The JSON format**: the planner must reply with a JSON list of strings. What does `parse_plan()` do if the planner replies with a chatty paragraph and no JSON list? Why is that failure mode dangerous (think about the execution loop)?

   `parse_plan()` looks for `[` and `]`; if neither exists (or `json.loads` fails on what's between them) it returns `[]`. Then `for step in plan:` iterates zero times, the DataAgent never runs, `step_answers` is empty, and the orchestrator returns an *empty string* as the answer — no exception, no warning. Silent emptiness is the worst failure mode because the trace looks "successful" (no error) while delivering nothing. Defense: assert `len(plan) >= 1` right after parsing.

5. **Cost**: planning adds one extra LLM call on top of one-per-sub-task. For a 2-step question that's 3 delegations vs Phase 1's 1. When is decomposition worth that cost, and when is it overkill?

   **Worth it** when the question genuinely splits into independent lookups, when sub-tasks can run in *parallel* (the planner overhead is amortized over wall-clock time saved), or when steps need different *tools* and you want the planner to route. **Overkill** for any single-lookup question — paying a planner call to "split" a question that doesn't need splitting is pure waste. A cheap heuristic: if the question contains "vs", "compare", "and then", "for each" → plan it. Otherwise call the DataAgent directly.

## Phase 3 done when:
- [ x] TODO 1 (PlannerAgent system prompt) filled in
- [ x] TODO 2 (execution loop in the orchestrator) filled in
- [ x] TODO 3 (your multi-step question) filled in
- [ x] TODO 4 (assertion) filled in
- [ x] TODO 5 (quiz) answered
- [ x] Cell 13 shows the planner producing >=2 sub-tasks
- [ x] Notebook runs top-to-bottom without errors
- [ x] `traces/traces.jsonl` has a new line with `n_delegations >= 3`

Then ping me — we'll review your planner prompt + quiz, then move to Phase 4 (hierarchical memory).